In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Read the dataset Q3_data.csv using read_csv()
import pandas as pd
import os

data_path = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(data_path)

In [ ]:
# Task 2: Inspect the first few rows using head()
df.head()

In [ ]:
# Task 3: Display dataset information using info()
df.info()

In [ ]:
# Task 4: Show statistical description using describe()
df.describe()

In [ ]:
# Task 1: Handle missing values appropriately

# Analyze missing values
def analyze_missing(df):
    missing_percentage = (df.isnull().sum() / len(df)) * 100

    missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
    })

    missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

    return missing_data

print("Missing Data Analysis:")
missing_data = analyze_missing(df)
missing_data

In [ ]:
# Task 2: Check and remove duplicates if any exist
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Encode categorical variables if needed
# The data has only float64(148), int64(41) types, no 'object' appeared in the data info

In [ ]:
# Task 4: Apply feature scaling to numerical features (Use StandardScaler)
from sklearn.preprocessing import StandardScaler

features = df.columns.drop('Target')

scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])
df.head()

In [ ]:
# Task 5: Check for target imbalance and state if it is imbalanced or not
import matplotlib.pyplot as plt

target = 'Target'
target_percentage = df[target].value_counts() / len(df) * 100
print(df[target].value_counts())
print()

plt.figure(figsize=(3, 3))
plt.bar(target_percentage.index, target_percentage.values, color='coral')
plt.title('Target Distribution')
plt.xlabel('Target')
plt.ylabel('Percentage')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Task 1: Split the dataset into features (X) and target (y)
X = df.drop('Target', axis=1)
y = df['Target']

In [ ]:
%%capture
!pip install catboost

In [ ]:
# Task 2: Use the correct split: KFold OR StratifiedKFold
from sklearn.model_selection import train_test_split, StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score

n_splits = 5
all_results = {'accuracy': [], 'f1': []}

# Task 3: Train a CatBoostClassifier model
model = CatBoostClassifier()

# I'll use StratifiedKFold as it's classification task with imbalanced data
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models
  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

  # Task 4: Evaluate using the appropriate metric only (Accuracy vs. F1 Score).
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, zero_division=0)

  all_results['accuracy'].append(accuracy)
  all_results['f1'].append(f1)

In [ ]:
# Task 5: Print the averaged score across all folds
import numpy as np

print(f"Accuracy:  {np.mean(all_results['accuracy']):.4f}")
print(f"F1-Score:  {np.mean(all_results['f1']):.4f}")

In [ ]:
# Task 1: Plot feature importance from your trained model
feature_importance = pd.DataFrame({
    'feature': df.columns.drop('Target'),
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 30))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
feature_importance.head()

In [ ]:
# Task 2: Identify and print the name of the most important feature (the 'golden feature')
feature, importance = feature_importance.items()
golden_feature = feature[1][0]
print("The most important feature (THE GOLDEN FEATURE) is:", golden_feature)

In [ ]:
# Task 1: Create new X with only the golden feature
X = df['P_2']
y = df['Target']

In [ ]:
# Task 2: Run the same KFold loop with this single feature
n_splits = 5
all_results = {'accuracy': [], 'f1': []}
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  # 1. Split data
  X_train, X_test = X.iloc[train_index].values.reshape(-1, 1), X.iloc[test_index].values.reshape(-1, 1)
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models
  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

  # 3. Evaluate
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, zero_division=0)

  all_results['accuracy'].append(accuracy)
  all_results['f1'].append(f1)

In [ ]:
# Task 3: Print and compare the accuracy with the full model
print(f"Accuracy:  {np.mean(all_results['accuracy']):.4f}")
print(f"F1-Score:  {np.mean(all_results['f1']):.4f}")